#  EuroSAT — CNN from Scratch (AlexNet-like)
**2nd Assignment — Deep Learning for Computer Vision**

In this notebook we design and train a convolutional neural network **from scratch** to classify EuroSAT satellite images into 10 land-use categories.

The architecture is inspired by **AlexNet** but adapted for EuroSAT's 64×64 input size.

**Goal:** Given a satellite image → predict one of 10 classes:
`AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, SeaLake`

## 0. Setup

In [1]:
!pip install torch torchvision matplotlib seaborn scikit-learn -q

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import EuroSAT

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import random, time, os

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cpu


## 1. Hyperparameters

In [3]:
# All hyperparameters in one place — easy to tune
CONFIG = {
    'batch_size'   : 64,
    'num_epochs'   : 30,
    'learning_rate': 1e-3,
    'weight_decay' : 1e-4,   # L2 regularization
    'num_classes'  : 10,
    'img_size'     : 64,
    # Normalization values computed in EDA
    'mean'         : [0.3444, 0.3803, 0.4078],
    'std'          : [0.2034, 0.1366, 0.1148],
    'train_split'  : 0.70,
    'val_split'    : 0.15,
    # test_split = 0.15 (remainder)
}

CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
    'River', 'SeaLake'
]

print('CONFIG:', CONFIG)

CONFIG: {'batch_size': 64, 'num_epochs': 30, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'num_classes': 10, 'img_size': 64, 'mean': [0.3444, 0.3803, 0.4078], 'std': [0.2034, 0.1366, 0.1148], 'train_split': 0.7, 'val_split': 0.15}


## 2. Data Loading & Augmentation

In [4]:
# Training: with data augmentation to improve generalization
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG['mean'], std=CONFIG['std'])
])

# Validation & Test: no augmentation, only normalize
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG['mean'], std=CONFIG['std'])
])

# Load full dataset (we'll split manually)
full_dataset = EuroSAT(root='./data', transform=train_transform, download=True)

total = len(full_dataset)
train_size = int(CONFIG['train_split'] * total)
val_size   = int(CONFIG['val_split']   * total)
test_size  = total - train_size - val_size

generator = torch.Generator().manual_seed(SEED)
train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)

# Apply val/test transform (no augmentation)
# We override the transform for val and test subsets
val_ds.dataset  = EuroSAT(root='./data', transform=val_transform, download=False)
test_ds.dataset = EuroSAT(root='./data', transform=val_transform, download=False)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train: 18,900 | Val: 4,050 | Test: 4,050
Train batches: 296 | Val batches: 64


## 3. Model Architecture — AlexNet-like CNN

AlexNet was designed for 224×224 images. Since EuroSAT images are 64×64, we adapt the architecture:
- Smaller kernel sizes and strides
- Fewer max-pooling layers to preserve spatial information
- Same conceptual structure: conv blocks → flatten → fully connected classifier

In [5]:
class AlexNetEuroSAT(nn.Module):
    """
    AlexNet-inspired CNN adapted for 64x64 EuroSAT images.
    
    Original AlexNet blocks:
      Conv(11,4) -> Conv(5,1) -> Conv(3,1)*3 -> FC(4096)*2 -> FC(1000)
    
    Our adaptation for 64x64:
      Conv(5,1) -> Conv(3,1) -> Conv(3,1)*3 -> FC(1024) -> FC(512) -> FC(num_classes)
    """
    def __init__(self, num_classes=10):
        super(AlexNetEuroSAT, self).__init__()
        
        # --- Feature Extractor (Convolutional Layers) ---
        self.features = nn.Sequential(
            # Block 1: input 3x64x64 -> 96x30x30
            nn.Conv2d(3, 96, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 96x32x32
            
            # Block 2: 96x32x32 -> 256x16x16
            nn.Conv2d(96, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 256x16x16
            
            # Block 3: 256x16x16 -> 384x16x16
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Block 4: 384x16x16 -> 384x16x16
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Block 5: 384x16x16 -> 256x8x8
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 256x8x8
        )
        
        # --- Classifier (Fully Connected Layers) ---
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 8 * 8, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)       # Extract spatial features
        x = x.view(x.size(0), -1)  # Flatten: (batch, 256*8*8)
        x = self.classifier(x)     # Classify
        return x


# Instantiate model
model = AlexNetEuroSAT(num_classes=CONFIG['num_classes']).to(DEVICE)

# Print architecture summary
print(model)
print()

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

AlexNetEuroSAT(
  (features): Sequential(
    (0): Conv2d(3, 96, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=16384, out_features=1024, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p

## 4. Loss Function, Optimizer & Scheduler

In [6]:
# CrossEntropyLoss — standard for multiclass classification
# It combines LogSoftmax + NLLLoss internally
criterion = nn.CrossEntropyLoss()

# Adam optimizer with L2 weight decay
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# StepLR: reduce LR by 0.1 every 10 epochs
# This helps the model converge better in later epochs
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

print('Loss:      CrossEntropyLoss')
print('Optimizer: Adam  (lr={}, weight_decay={})'.format(
    CONFIG['learning_rate'], CONFIG['weight_decay']))
print('Scheduler: StepLR (step=10, gamma=0.1)')

Loss:      CrossEntropyLoss
Optimizer: Adam  (lr=0.001, weight_decay=0.0001)
Scheduler: StepLR (step=10, gamma=0.1)


## 5. Training & Validation Loop

In [7]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Run one full training epoch. Returns avg loss and accuracy."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()          # Clear gradients
        outputs = model(imgs)          # Forward pass
        loss = criterion(outputs, labels)  # Compute loss
        loss.backward()                # Backpropagation
        optimizer.step()               # Update weights
        
        running_loss += loss.item() * imgs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
    
    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate model on a dataloader. Returns avg loss and accuracy."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():  # No gradient computation needed
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * imgs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    
    return running_loss / total, correct / total

In [8]:
# Training loop
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_model_path = 'best_cnn_scratch.pth'

print(f'Training for {CONFIG["num_epochs"]} epochs on {DEVICE}...')
print('-' * 65)
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"LR":>8}')
print('-' * 65)

start_time = time.time()

for epoch in range(1, CONFIG['num_epochs'] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        flag = ' ✅'
    else:
        flag = ''
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f'{epoch:>5} | {train_loss:>10.4f} | {train_acc*100:>8.2f}% | '
          f'{val_loss:>8.4f} | {val_acc*100:>6.2f}%{flag} | {current_lr:.2e}')

elapsed = time.time() - start_time
print('-' * 65)
print(f'Training complete in {elapsed/60:.1f} min')
print(f'Best validation accuracy: {best_val_acc*100:.2f}%')

Training for 30 epochs on cpu...
-----------------------------------------------------------------
Epoch | Train Loss | Train Acc | Val Loss | Val Acc |       LR
-----------------------------------------------------------------


/home/andonigarrido/DeepLearningCourse/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


    1 |     1.5057 |    40.88% |   1.1323 |  55.93% ✅ | 1.00e-03
    2 |     0.9773 |    64.93% |   0.8173 |  70.74% ✅ | 1.00e-03
    3 |     0.8516 |    69.84% |   0.6841 |  76.27% ✅ | 1.00e-03
    4 |     0.7843 |    71.90% |   0.6556 |  76.12% | 1.00e-03


KeyboardInterrupt: 

## 6. Training Curves

In [ ]:
epochs = range(1, CONFIG['num_epochs'] + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN from Scratch — Training Curves', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='#e74c3c', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='#3498db', linewidth=2)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train Acc',
             color='#e74c3c', linewidth=2)
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   label='Val Acc',
             color='#3498db', linewidth=2)
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_scratch_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Overfitting check
gap = history['train_acc'][-1] - history['val_acc'][-1]
print(f'Final train acc: {history["train_acc"][-1]*100:.2f}%')
print(f'Final val acc:   {history["val_acc"][-1]*100:.2f}%')
print(f'Gap: {gap*100:.2f}% → {"⚠️ Possible overfitting" if gap > 0.1 else "✅ Good generalization"}')

## 7. Test Set Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)

print('=' * 40)
print('       TEST SET RESULTS')
print('=' * 40)
print(f'  Test Loss:     {test_loss:.4f}')
print(f'  Test Accuracy: {test_acc*100:.2f}%')
print('=' * 40)

## 8. Confusion Matrix

In [ ]:
# Collect all predictions and true labels
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('CNN from Scratch — Confusion Matrix', fontsize=14, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0])
axes[0].set_title('Raw Counts')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
axes[0].tick_params(axis='x', rotation=45)

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[1])
axes[1].set_title('Normalized (per true class)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('cnn_scratch_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Per-Class Report

In [ ]:
print('Per-class Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

## 10. Visualize Predictions on Sample Images

In [ ]:
# Show 16 random test images with predicted vs true label
model.eval()
imgs_batch, labels_batch = next(iter(test_loader))
imgs_batch = imgs_batch[:16]
labels_batch = labels_batch[:16]

with torch.no_grad():
    outputs = model(imgs_batch.to(DEVICE))
    _, preds = outputs.max(1)

# Denormalize for display
mean = torch.tensor(CONFIG['mean']).view(3,1,1)
std  = torch.tensor(CONFIG['std']).view(3,1,1)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Sample Predictions — CNN from Scratch', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    img = imgs_batch[i] * std + mean  # denormalize
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    
    true_label = CLASS_NAMES[labels_batch[i]]
    pred_label = CLASS_NAMES[preds[i].cpu()]
    correct = true_label == pred_label
    
    ax.imshow(img)
    ax.set_title(f'T: {true_label}\nP: {pred_label}',
                 fontsize=7,
                 color='green' if correct else 'red',
                 fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('cnn_scratch_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Green = correct prediction | Red = wrong prediction')

## 11. Summary

In [ ]:
print('=' * 50)
print('    CNN FROM SCRATCH — FINAL SUMMARY')
print('=' * 50)
print(f'  Architecture   : AlexNet-like (adapted for 64x64)')
print(f'  Parameters     : {trainable_params:,}')
print(f'  Epochs trained : {CONFIG["num_epochs"]}')
print(f'  Training time  : {elapsed/60:.1f} min')
print(f'  Best Val Acc   : {best_val_acc*100:.2f}%')
print(f'  Test Accuracy  : {test_acc*100:.2f}%')
print(f'  Test Loss      : {test_loss:.4f}')
print()
print('  → These results will be compared against')
print('    Transfer Learning in the next notebook.')
print('=' * 50)